# Fashion MNIST Classification with Convolutional Neural Networks

**Course:** Neural Networks — Final Project  
**Institution:** Instituto Tecnológico y de Estudios Superiores de Monterrey  

## **Team 2**

### Cesar Castaño A00830006
### Félix Martínez A01284040
### Raúl Valdés Caballero A01197480
### Diego Axel Marquez A01707739
---

## Objective

Train a classification model on the **Fashion MNIST** dataset using progressive CNN architectures, demonstrating key concepts: regularization, optimization, hyperparameter tuning, and architecture design.

### Roadmap
1. **Setup & Data Loading** — Environment, EDA, preprocessing
2. **Model 1: Simple Baseline CNN** — Bare-bones CNN, no regularization
3. **Model 2: Improved Custom CNN** — BatchNorm, Dropout, Data Augmentation, LR Scheduling
4. **Model 3: Mini-ResNet** — Residual connections, advanced optimization
5. **Hyperparameter Tuning** — Manual grid search on best model
6. **Final Comparison & Results** — Side-by-side analysis
7. **Conclusions & References**

---
## 1. Setup & Data Loading

In [ ]:
# Install dependencies (Colab-friendly)
!pip install -q kagglehub seaborn scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import time
import os
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# REPRODUCIBILITY CONFIGURATION
# ============================================================
# Set SEED to an integer (e.g., 42) for reproducible results.
# Set SEED = None to get different results on each run.
# ============================================================
SEED = 42

if SEED is not None:
    os.environ["PYTHONHASHSEED"] = str(SEED)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"         # Force deterministic GPU ops
    os.environ["TF_CUDNN_DETERMINISTIC"] = "1"        # Deterministic cuDNN
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    print(f"Reproducibility ENABLED — SEED = {SEED}")
    print("  → Set SEED = None above to get different results on each run.")
else:
    print("Reproducibility DISABLED — results will vary between runs.")
    print("  → Set SEED = 42 (or any integer) above to get reproducible results.")

# Verify GPU availability (important for Colab)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices("GPU")}")

In [ ]:
# Load Fashion MNIST from Kaggle
import kagglehub

path = kagglehub.dataset_download("zalando-research/fashionmnist")
print(f"Dataset downloaded to: {path}")

In [ ]:
import os

# Load CSV files
train_df = pd.read_csv(os.path.join(path, 'fashion-mnist_train.csv'))
test_df = pd.read_csv(os.path.join(path, 'fashion-mnist_test.csv'))

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"\nColumns: {train_df.columns[:5].tolist()} ... ({len(train_df.columns)} total)")
train_df.head()

In [ ]:
# Class names for Fashion MNIST
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Separate features and labels
y_train_full = train_df['label'].values
X_train_full = train_df.drop('label', axis=1).values.reshape(-1, 28, 28, 1)

y_test = test_df['label'].values
X_test = test_df.drop('label', axis=1).values.reshape(-1, 28, 28, 1)

# Normalize pixel values to [0, 1]
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Split training into train + validation (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=SEED, stratify=y_train_full
)

print(f"Training set:   {X_train.shape} | Labels: {y_train.shape}")
print(f"Validation set: {X_val.shape}  | Labels: {y_val.shape}")
print(f"Test set:       {X_test.shape}  | Labels: {y_test.shape}")

### Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title in zip(axes, [y_train, y_test], ['Training Set', 'Test Set']):
    unique, counts = np.unique(data, return_counts=True)
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    bars = ax.bar([CLASS_NAMES[i] for i in unique], counts, color=colors)
    ax.set_title(f'Class Distribution — {title}', fontsize=13)
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                str(count), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx].squeeze(), cmap='gray')
    ax.set_title(CLASS_NAMES[i], fontsize=11)
    ax.axis('off')

plt.suptitle('Sample Images — One per Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pixel intensity distribution
fig, axes = plt.subplots(2, 5, figsize=(16, 6))

for i, ax in enumerate(axes.flat):
    class_images = X_train[y_train == i]
    mean_image = class_images.mean(axis=0).squeeze()
    ax.imshow(mean_image, cmap='hot')
    ax.set_title(f'{CLASS_NAMES[i]}', fontsize=10)
    ax.axis('off')

plt.suptitle('Average Image per Class (Heat Map)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Overall pixel statistics
print(f"Pixel value range: [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"Mean pixel value:  {X_train.mean():.4f}")
print(f"Std pixel value:   {X_train.std():.4f}")
print(f"\nImage shape: {X_train[0].shape}")
print(f"Number of classes: {len(CLASS_NAMES)}")

### Helper Functions

We define reusable functions for training visualization and evaluation that will be used across all models.

In [ ]:
def plot_training_history(history, title='Training History'):
    """Plot accuracy and loss curves for training and validation."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    ax1.set_title(f'{title} — Accuracy', fontsize=13)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'], label='Train', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Validation', linewidth=2)
    ax2.set_title(f'{title} — Loss', fontsize=13)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def evaluate_model(model, X_test, y_test, model_name='Model'):
    """Evaluate model and display confusion matrix + classification report."""
    y_pred = model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)

    # Test accuracy
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    f1 = f1_score(y_test, y_pred_classes, average='weighted')
    print(f"\n{'='*50}")
    print(f"{model_name} — Test Results")
    print(f"{'='*50}")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Loss:     {test_loss:.4f}")
    print(f"Weighted F1:   {f1:.4f}")

    # Classification report
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=CLASS_NAMES))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_classes)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix', fontsize=14)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return test_acc, test_loss, f1, y_pred_classes

---
## 2. Model 1 — Simple Baseline CNN

Our first model is a bare-bones CNN with **no regularization techniques**. This serves as a baseline to measure the impact of improvements we introduce later.

**Architecture:**
- 2 Convolutional layers (32 and 64 filters, 3×3 kernels)
- MaxPooling after each conv layer
- Flatten → Dense(128) → Dense(10, softmax)
- Optimizer: SGD (lr=0.01)
- No Dropout, no BatchNorm, no data augmentation

In [ ]:
def build_baseline_cnn():
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=0.01),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

baseline_model = build_baseline_cnn()
baseline_model.summary()

In [ ]:
# Train baseline model
EPOCHS_BASELINE = 25
BATCH_SIZE = 64

start_time = time.time()
baseline_history = baseline_model.fit(
    X_train, y_train,
    epochs=EPOCHS_BASELINE,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1
)
baseline_train_time = time.time() - start_time
print(f"\nTraining time: {baseline_train_time:.1f}s")

In [ ]:
plot_training_history(baseline_history, title='Model 1: Baseline CNN')

In [ ]:
baseline_acc, baseline_loss, baseline_f1, baseline_preds = evaluate_model(
    baseline_model, X_test, y_test, 'Model 1: Baseline CNN'
)

### Baseline Observations

**Key takeaways from the baseline model:**
- The model achieves a reasonable accuracy with minimal architecture
- Notice the gap between training and validation accuracy — this indicates **overfitting**
- The loss curves likely diverge after several epochs
- Certain classes (e.g., Shirt vs T-shirt/top) are frequently confused

These observations motivate our next model where we add regularization techniques.

---
## 3. Model 2 — Improved Custom CNN

We now address the baseline's weaknesses by applying key concepts from class:

| Technique | Purpose |
|-----------|----------|
| **Batch Normalization** | Stabilize and accelerate training |
| **Dropout** | Regularization — prevent overfitting |
| **Data Augmentation** | Regularization — increase effective dataset size |
| **L2 Weight Decay** | Regularization — penalize large weights |
| **Adam Optimizer** | Adaptive learning rate for faster convergence |
| **LR Scheduling** | ReduceLROnPlateau — lower LR when progress stalls |
| **Early Stopping** | Prevent overfitting by stopping at optimal epoch |

In [ ]:
# Data augmentation layer (Keras preprocessing layers — Colab compatible)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name='data_augmentation')

# Visualize augmented samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
sample_image = X_train[0:1]  # Single image batch

for ax in axes.flat:
    augmented = data_augmentation(sample_image, training=True)
    ax.imshow(augmented[0].numpy().squeeze(), cmap='gray')
    ax.axis('off')

plt.suptitle(f'Data Augmentation Examples — {CLASS_NAMES[y_train[0]]}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def build_improved_cnn(l2_reg=1e-4, dropout_conv=0.25, dropout_dense=0.5):
    model = models.Sequential([
        layers.Input(shape=(28, 28, 1)),

        # Data augmentation (only active during training)
        data_augmentation,

        # Block 1
        layers.Conv2D(32, (3, 3), padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3, 3), padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_conv),

        # Block 2
        layers.Conv2D(64, (3, 3), padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3, 3), padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_conv),

        # Block 3
        layers.Conv2D(128, (3, 3), padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_conv),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, kernel_regularizer=regularizers.l2(l2_reg)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(dropout_dense),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

improved_model = build_improved_cnn()
improved_model.summary()

In [ ]:
# Callbacks
improved_callbacks = [
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=7, restore_best_weights=True, verbose=1
    )
]

# Train improved model
EPOCHS_IMPROVED = 50

start_time = time.time()
improved_history = improved_model.fit(
    X_train, y_train,
    epochs=EPOCHS_IMPROVED,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=improved_callbacks,
    verbose=1
)
improved_train_time = time.time() - start_time
print(f"\nTraining time: {improved_train_time:.1f}s")

In [ ]:
plot_training_history(improved_history, title='Model 2: Improved CNN')

In [ ]:
improved_acc, improved_loss, improved_f1, improved_preds = evaluate_model(
    improved_model, X_test, y_test, 'Model 2: Improved CNN'
)

### Improved CNN Observations

**Compared to baseline:**
- The train/validation gap should be **significantly reduced** (less overfitting)
- Data augmentation helps the model generalize to unseen variations
- BatchNorm stabilizes training, allowing faster convergence with Adam
- LR scheduling and early stopping prevent wasted epochs
- The confusion between similar classes (Shirt/T-shirt, Coat/Pullover) should be reduced

---
## 4. Model 3 — Mini-ResNet (Residual Network)

Our most advanced model introduces **residual connections** (skip connections), the key innovation from He et al. (2015). These connections:

- Allow gradients to flow directly through the network, mitigating the **vanishing gradient problem**
- Enable training of deeper networks
- Act as an implicit form of **regularization**

We implement a custom mini-ResNet adapted for the 28×28 Fashion MNIST images using the Keras Functional API.

In [ ]:
def residual_block(x, filters, stride=1, l2_reg=1e-4):
    """A single residual block with two conv layers and a skip connection."""
    shortcut = x

    # First convolution
    x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second convolution
    x = layers.Conv2D(filters, (3, 3), strides=1, padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    # Adjust shortcut dimensions if needed
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), strides=stride, padding='same',
                                 kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Skip connection
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x


def build_mini_resnet(l2_reg=1e-4, dropout_rate=0.4):
    """Build a mini-ResNet adapted for 28x28 grayscale images."""
    inputs = layers.Input(shape=(28, 28, 1))

    # Data augmentation
    x = data_augmentation(inputs)

    # Initial convolution
    x = layers.Conv2D(64, (3, 3), padding='same',
                      kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Residual blocks — Stage 1 (64 filters, 28x28)
    x = residual_block(x, 64, stride=1, l2_reg=l2_reg)
    x = residual_block(x, 64, stride=1, l2_reg=l2_reg)
    x = layers.Dropout(dropout_rate / 2)(x)

    # Residual blocks — Stage 2 (128 filters, 14x14)
    x = residual_block(x, 128, stride=2, l2_reg=l2_reg)
    x = residual_block(x, 128, stride=1, l2_reg=l2_reg)
    x = layers.Dropout(dropout_rate / 2)(x)

    # Residual blocks — Stage 3 (256 filters, 7x7)
    x = residual_block(x, 256, stride=2, l2_reg=l2_reg)
    x = residual_block(x, 256, stride=1, l2_reg=l2_reg)
    x = layers.Dropout(dropout_rate)(x)

    # Global Average Pooling (replaces Flatten + large Dense layers)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(10, activation='softmax')(x)

    model = models.Model(inputs, x, name='Mini_ResNet')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

resnet_model = build_mini_resnet()
resnet_model.summary()

In [ ]:
# ── GPU check: if this fails, go to Runtime → Change runtime type → T4 GPU ──
gpus = tf.config.list_physical_devices("GPU")
if len(gpus) == 0:
    raise RuntimeError("No GPU detected! Fix: Runtime → Change runtime type → T4 GPU → Save → Reconnect → Re-run all.")
print(f"GPU ready: {gpus[0].name}")

# Learning rate schedule: cosine decay
EPOCHS_RESNET = 40          # EarlyStopping will stop before this if needed
BATCH_SIZE_RESNET = 128     # Larger batch → faster GPU utilization

cosine_decay = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=EPOCHS_RESNET * (len(X_train) // BATCH_SIZE_RESNET),
    alpha=1e-6
)

resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cosine_decay),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

resnet_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
    )
]

start_time = time.time()
resnet_history = resnet_model.fit(
    X_train, y_train,
    epochs=EPOCHS_RESNET,
    batch_size=BATCH_SIZE_RESNET,
    validation_data=(X_val, y_val),
    callbacks=resnet_callbacks,
    verbose=1
)
resnet_train_time = time.time() - start_time
print(f"Training time: {resnet_train_time:.1f}s ({resnet_train_time/60:.1f} min)")


In [ ]:
plot_training_history(resnet_history, title='Model 3: Mini-ResNet')

In [ ]:
resnet_acc, resnet_loss, resnet_f1, resnet_preds = evaluate_model(
    resnet_model, X_test, y_test, 'Model 3: Mini-ResNet'
)

### Mini-ResNet Observations

**Key advantages of the residual architecture:**
- Skip connections enable effective training of a deeper network (13 conv layers)
- GlobalAveragePooling replaces Flatten+Dense, reducing parameters and overfitting
- Cosine decay provides a smooth LR schedule that helps find better minima
- The model should achieve the highest accuracy among our three architectures

---
## 5. Hyperparameter Tuning — Manual Grid Search

We perform a manual grid search on our best architecture (Mini-ResNet) to find optimal hyperparameters.

**Parameters to tune:**
- Learning rate: `[1e-2, 1e-3, 1e-4]`
- Batch size: `[32, 64, 128]`
- Dropout rate: `[0.3, 0.4, 0.5]`

To keep training time manageable, we use a **subset of combinations** and train for fewer epochs as a screening pass.

In [ ]:
# Hyperparameter grid (selected combinations to keep time manageable)
hp_configs = [
    {'lr': 1e-2, 'batch_size': 64,  'dropout': 0.4},
    {'lr': 1e-3, 'batch_size': 32,  'dropout': 0.3},
    {'lr': 1e-3, 'batch_size': 64,  'dropout': 0.4},
    {'lr': 1e-3, 'batch_size': 64,  'dropout': 0.5},
    {'lr': 1e-3, 'batch_size': 128, 'dropout': 0.4},
    {'lr': 1e-4, 'batch_size': 64,  'dropout': 0.4},
    {'lr': 1e-3, 'batch_size': 64,  'dropout': 0.3},
    {'lr': 1e-4, 'batch_size': 32,  'dropout': 0.5},
]

SCREENING_EPOCHS = 20
hp_results = []

print(f"Running grid search: {len(hp_configs)} configurations x {SCREENING_EPOCHS} epochs")
print("=" * 70)

for i, config in enumerate(hp_configs):
    print(f"\n[{i+1}/{len(hp_configs)}] LR={config['lr']}, BS={config['batch_size']}, Dropout={config['dropout']}")

    # Build model with current hyperparameters
    hp_model = build_mini_resnet(dropout_rate=config['dropout'])
    hp_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    hp_history = hp_model.fit(
        X_train, y_train,
        epochs=SCREENING_EPOCHS,
        batch_size=config['batch_size'],
        validation_data=(X_val, y_val),
        callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
        verbose=0
    )

    val_loss, val_acc = hp_model.evaluate(X_val, y_val, verbose=0)
    print(f"   → Val Accuracy: {val_acc:.4f} | Val Loss: {val_loss:.4f}")

    hp_results.append({
        'lr': config['lr'],
        'batch_size': config['batch_size'],
        'dropout': config['dropout'],
        'val_accuracy': val_acc,
        'val_loss': val_loss
    })

print("\nGrid search complete!")

In [ ]:
# Results table
hp_df = pd.DataFrame(hp_results).sort_values('val_accuracy', ascending=False)
hp_df.index = range(1, len(hp_df) + 1)
hp_df.index.name = 'Rank'
print("Hyperparameter Tuning Results (sorted by validation accuracy):")
print("=" * 70)
hp_df

In [ ]:
# Visualize grid search results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by learning rate
for lr in sorted(hp_df['lr'].unique()):
    subset = hp_df[hp_df['lr'] == lr]
    axes[0].scatter([str(lr)] * len(subset), subset['val_accuracy'], s=100, label=f'LR={lr}')
axes[0].set_title('Validation Accuracy by Learning Rate', fontsize=13)
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Validation Accuracy')
axes[0].grid(True, alpha=0.3)

# Heatmap: LR vs Dropout (for batch_size=64)
pivot_data = hp_df[hp_df['batch_size'] == 64].pivot_table(
    values='val_accuracy', index='dropout', columns='lr'
)
if not pivot_data.empty:
    sns.heatmap(pivot_data, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[1])
    axes[1].set_title('Val Accuracy: Dropout vs LR (BS=64)', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# Retrain final model with best hyperparameters
best_config = hp_df.iloc[0]
print(f"Best configuration:")
print(f"  Learning Rate: {best_config['lr']}")
print(f"  Batch Size:    {int(best_config['batch_size'])}")
print(f"  Dropout:       {best_config['dropout']}")
print(f"  Val Accuracy:  {best_config['val_accuracy']:.4f}")

# Build and train final model with best params + full epochs
best_lr = best_config['lr']
best_bs = int(best_config['batch_size'])
best_dropout = best_config['dropout']

EPOCHS_FINAL = 60

final_model = build_mini_resnet(dropout_rate=best_dropout)

cosine_decay_final = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=best_lr,
    decay_steps=EPOCHS_FINAL * (len(X_train) // best_bs),
    alpha=1e-6
)

final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=cosine_decay_final),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

final_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )
]

start_time = time.time()
final_history = final_model.fit(
    X_train, y_train,
    epochs=EPOCHS_FINAL,
    batch_size=best_bs,
    validation_data=(X_val, y_val),
    callbacks=final_callbacks,
    verbose=1
)
final_train_time = time.time() - start_time
print(f"\nFinal model training time: {final_train_time:.1f}s")

In [ ]:
plot_training_history(final_history, title='Final Model: Tuned Mini-ResNet')

In [ ]:
final_acc, final_loss, final_f1, final_preds = evaluate_model(
    final_model, X_test, y_test, 'Final Model: Tuned Mini-ResNet'
)

---
## 6. Final Comparison & Results

In [ ]:
# Summary comparison table
results_data = {
    'Model': ['Baseline CNN', 'Improved CNN', 'Mini-ResNet', 'Tuned Mini-ResNet'],
    'Parameters': [
        baseline_model.count_params(),
        improved_model.count_params(),
        resnet_model.count_params(),
        final_model.count_params()
    ],
    'Test Accuracy': [baseline_acc, improved_acc, resnet_acc, final_acc],
    'Test Loss': [baseline_loss, improved_loss, resnet_loss, final_loss],
    'Weighted F1': [baseline_f1, improved_f1, resnet_f1, final_f1],
    'Train Time (s)': [
        round(baseline_train_time, 1),
        round(improved_train_time, 1),
        round(resnet_train_time, 1),
        round(final_train_time, 1)
    ]
}

results_df = pd.DataFrame(results_data)
print("Model Comparison Summary")
print("=" * 90)
results_df

In [ ]:
# Overlaid accuracy comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

histories = [
    (baseline_history, 'Baseline CNN', '-'),
    (improved_history, 'Improved CNN', '--'),
    (resnet_history, 'Mini-ResNet', '-.'),
    (final_history, 'Tuned Mini-ResNet', ':'),
]

colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

for (hist, name, ls), color in zip(histories, colors):
    ax1.plot(hist.history['val_accuracy'], label=name, linestyle=ls, linewidth=2, color=color)
    ax2.plot(hist.history['val_loss'], label=name, linestyle=ls, linewidth=2, color=color)

ax1.set_title('Validation Accuracy Comparison', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.set_title('Validation Loss Comparison', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_names = results_df['Model']
x = np.arange(len(model_names))

# Accuracy
bars = axes[0].bar(x, results_df['Test Accuracy'], color=colors, alpha=0.85)
axes[0].set_title('Test Accuracy', fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=20, ha='right')
axes[0].set_ylim(0.85, 1.0)
for bar, val in zip(bars, results_df['Test Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# F1 Score
bars = axes[1].bar(x, results_df['Weighted F1'], color=colors, alpha=0.85)
axes[1].set_title('Weighted F1 Score', fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=20, ha='right')
axes[1].set_ylim(0.85, 1.0)
for bar, val in zip(bars, results_df['Weighted F1']):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# Parameters
bars = axes[2].bar(x, results_df['Parameters'] / 1000, color=colors, alpha=0.85)
axes[2].set_title('Model Size (K params)', fontsize=13)
axes[2].set_xticks(x)
axes[2].set_xticklabels(model_names, rotation=20, ha='right')
for bar, val in zip(bars, results_df['Parameters'] / 1000):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                 f'{val:.0f}K', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy analysis (final model)
cm_final = confusion_matrix(y_test, final_preds)
per_class_acc = cm_final.diagonal() / cm_final.sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(CLASS_NAMES, per_class_acc, color=plt.cm.RdYlGn(per_class_acc))
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Per-Class Accuracy — Tuned Mini-ResNet', fontsize=14)
ax.set_xlim(0.8, 1.0)
ax.grid(True, alpha=0.3, axis='x')

for bar, acc in zip(bars, per_class_acc):
    ax.text(acc + 0.003, bar.get_y() + bar.get_height()/2.,
            f'{acc:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nHardest classes (lowest accuracy):")
sorted_idx = np.argsort(per_class_acc)
for idx in sorted_idx[:3]:
    print(f"  {CLASS_NAMES[idx]}: {per_class_acc[idx]:.4f}")

In [ ]:
# Sample predictions: correct, incorrect, most/least confident
y_pred_probs = final_model.predict(X_test, verbose=0)
y_pred_labels = np.argmax(y_pred_probs, axis=1)
y_pred_confidence = np.max(y_pred_probs, axis=1)

correct_mask = y_pred_labels == y_test
incorrect_mask = ~correct_mask

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

# Top 5 most confident correct predictions
correct_conf = y_pred_confidence.copy()
correct_conf[incorrect_mask] = 0
top_correct = np.argsort(correct_conf)[-5:][::-1]

for i, idx in enumerate(top_correct):
    axes[0, i].imshow(X_test[idx].squeeze(), cmap='gray')
    axes[0, i].set_title(f'{CLASS_NAMES[y_test[idx]]}\nConf: {y_pred_confidence[idx]:.3f}',
                         fontsize=9, color='green')
    axes[0, i].axis('off')

# Top 5 most confident WRONG predictions
wrong_conf = y_pred_confidence.copy()
wrong_conf[correct_mask] = 0
top_wrong = np.argsort(wrong_conf)[-5:][::-1]

for i, idx in enumerate(top_wrong):
    axes[1, i].imshow(X_test[idx].squeeze(), cmap='gray')
    axes[1, i].set_title(f'True: {CLASS_NAMES[y_test[idx]]}\nPred: {CLASS_NAMES[y_pred_labels[idx]]}\nConf: {y_pred_confidence[idx]:.3f}',
                         fontsize=8, color='red')
    axes[1, i].axis('off')

axes[0, 0].annotate('Most Confident\nCorrect', xy=(-0.3, 0.5), xycoords='axes fraction',
                     fontsize=12, fontweight='bold', color='green', va='center', ha='right')
axes[1, 0].annotate('Most Confident\nIncorrect', xy=(-0.3, 0.5), xycoords='axes fraction',
                     fontsize=12, fontweight='bold', color='red', va='center', ha='right')

plt.suptitle('Sample Predictions — Tuned Mini-ResNet', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# t-SNE visualization of learned features
from sklearn.manifold import TSNE

# Extract features from the penultimate layer (before softmax)
feature_extractor = models.Model(
    inputs=final_model.input,
    outputs=final_model.layers[-2].output  # GlobalAveragePooling output
)

# Use a subset for t-SNE (it's computationally expensive)
n_samples = 3000
indices = np.random.choice(len(X_test), n_samples, replace=False)
X_subset = X_test[indices]
y_subset = y_test[indices]

features = feature_extractor.predict(X_subset, verbose=0)
print(f"Feature shape: {features.shape}")

# Run t-SNE
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(features)

# Plot
plt.figure(figsize=(12, 10))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1],
                      c=y_subset, cmap='tab10', alpha=0.6, s=10)

# Create legend
handles = [plt.scatter([], [], c=plt.cm.tab10(i/10), s=50, label=CLASS_NAMES[i]) for i in range(10)]
plt.legend(handles=handles, loc='best', fontsize=9)

plt.title('t-SNE Visualization of Learned Features (Tuned Mini-ResNet)', fontsize=14)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 7. Conclusions

### Progressive Improvement Summary

| Model | Key Additions | Expected Improvement |
|-------|--------------|---------------------|
| **Baseline CNN** | Basic Conv+Pool+Dense | Baseline reference |
| **Improved CNN** | BatchNorm, Dropout, Augmentation, Adam, LR schedule | Reduced overfitting, higher accuracy |
| **Mini-ResNet** | Residual connections, deeper architecture, cosine decay | Better feature learning, highest accuracy |
| **Tuned Mini-ResNet** | Hyperparameter optimization | Fine-tuned peak performance |

### Key Takeaways

1. **Regularization matters**: Dropout, BatchNorm, and data augmentation collectively reduced the overfitting gap significantly
2. **Architecture design**: Residual connections enabled training of a deeper network without degradation
3. **Optimization**: Moving from SGD to Adam with learning rate scheduling accelerated convergence
4. **Hyperparameter tuning**: Even small adjustments to learning rate, batch size, and dropout can yield meaningful improvements
5. **Hardest classes**: Shirt, T-shirt/top, and Coat are consistently the most confused categories due to visual similarity

---
## References

1. **Fashion MNIST Dataset**: Xiao, H., Rasul, K., & Vollgraf, R. (2017). *Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms*. arXiv:1708.07747. https://arxiv.org/abs/1708.07747

2. **Residual Networks (ResNet)**: He, K., Zhang, X., Ren, S., & Sun, J. (2016). *Deep Residual Learning for Image Recognition*. In Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR), pp. 770-778. https://arxiv.org/abs/1512.03385

3. **Batch Normalization**: Ioffe, S., & Szegedy, C. (2015). *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*. In Proceedings of the 32nd International Conference on Machine Learning (ICML), pp. 448-456. https://arxiv.org/abs/1502.03167

4. **Dropout**: Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. Journal of Machine Learning Research, 15(1), pp. 1929-1958. https://jmlr.org/papers/v15/srivastava14a.html

5. **Adam Optimizer**: Kingma, D. P., & Ba, J. (2015). *Adam: A Method for Stochastic Optimization*. In Proceedings of the 3rd International Conference on Learning Representations (ICLR). https://arxiv.org/abs/1412.6980

6. **Data Augmentation**: Shorten, C., & Khoshgoftaar, T. M. (2019). *A survey on Image Data Augmentation for Deep Learning*. Journal of Big Data, 6(1), pp. 1-48. https://doi.org/10.1186/s40537-019-0197-0

7. **Cosine Annealing LR Schedule**: Loshchilov, I., & Hutter, F. (2017). *SGDR: Stochastic Gradient Descent with Warm Restarts*. In Proceedings of the 5th International Conference on Learning Representations (ICLR). https://arxiv.org/abs/1608.03983

8. **L2 Regularization (Weight Decay)**: Krogh, A., & Hertz, J. A. (1991). *A Simple Weight Decay Can Improve Generalization*. In Advances in Neural Information Processing Systems (NeurIPS), pp. 950-957.

9. **t-SNE Visualization**: van der Maaten, L., & Hinton, G. (2008). *Visualizing Data using t-SNE*. Journal of Machine Learning Research, 9, pp. 2579-2605. https://jmlr.org/papers/v9/vandermaaten08a.html

10. **TensorFlow/Keras**: Abadi, M., et al. (2015). *TensorFlow: Large-Scale Machine Learning on Heterogeneous Systems*. https://www.tensorflow.org/

11. **Global Average Pooling**: Lin, M., Chen, Q., & Yan, S. (2014). *Network In Network*. In Proceedings of the 2nd International Conference on Learning Representations (ICLR). https://arxiv.org/abs/1312.4400